<a href="https://colab.research.google.com/github/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/blob/main/telco_bronze_to_silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# telco_bronze_to_silver.ipynb
**Dự án:** Telco Customer Churn – Medallion (Bronze → Silver) với BigQuery + dbt

| Phần | Nội dung | Phụ trách |
|---|---|---|
| 1 | Khảo sát & profiling dataset (21 cột) | Phân tích dataset & Bronze |
| 2 | Nạp CSV vào `bronze_telco.customers_raw` | Phân tích dataset & Bronze |
| 3 | dbt staging / Silver (`stg_telco_customers`, `customers_cleaned`) | Thành viên dbt |

Quy tắc đặt tên: file gốc `telco_customer_churn.csv` · Bronze `bronze_telco.customers_raw` · Silver `silver_telco.customers_cleaned`.

Cell 1 — Cấu hình và xác thực Google Cloud

In [5]:

PROJECT_ID  = "btqt2-telco-dbt-bigquery"
BQ_LOCATION = "asia-southeast1"       # giữ cùng location cho Bronze và Silver
BRONZE_DATASET = "bronze_telco"
BRONZE_TABLE   = "customers_raw"
SOURCE_FILE    = "telco_customer_churn.csv"

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
print("Đã kết nối BigQuery:", bq.project)


Đã kết nối BigQuery: btqt2-telco-dbt-bigquery


Cell 2 — Kiểm tra kết nối và tạo Bronze/Silver datasets

In [12]:

import os

SILVER_DATASET = "silver_telco"
SILVER_TABLE = "customers_cleaned"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

!gcloud config set project {PROJECT_ID} --quiet

print("Project:", bq.project)
print(
    "Kiểm tra kết nối:",
    list(bq.query("SELECT 1 AS ok").result())[0].ok
)

for name, desc in [
    (
        BRONZE_DATASET,
        "Bronze layer - du lieu Telco Customer Churn goc"
    ),
    (
        SILVER_DATASET,
        "Silver layer - du lieu da lam sach bang dbt"
    ),
]:
    d = bigquery.Dataset(f"{PROJECT_ID}.{name}")
    d.location = BQ_LOCATION
    d.description = desc

    d = bq.create_dataset(d, exists_ok=True)
    print("Dataset:", d.full_dataset_id, "| Location:", d.location)

print(
    "Danh sach dataset:",
    [d.dataset_id for d in bq.list_datasets()]
)


[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Project: btqt2-telco-dbt-bigquery
Kiểm tra kết nối: 1
Dataset: btqt2-telco-dbt-bigquery:bronze_telco | Location: asia-southeast1
Dataset: btqt2-telco-dbt-bigquery:silver_telco | Location: asia-southeast1
Danh sach dataset: ['bronze_telco', 'silver_telco']


Cell 3 — Upload CSV và đọc dữ liệu *gốc*

In [13]:

import os
import pandas as pd
from urllib.request import urlretrieve

SOURCE_FILE = "telco_customer_churn.csv"

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "huyenpiano1711/BTQT2_Telco_dbt_BigQuery/"
    "main/telco_customer_churn.csv"
)

urlretrieve(DATA_URL, SOURCE_FILE)

df = pd.read_csv(
    SOURCE_FILE,
    dtype=str,
    keep_default_na=False
)

print("Nguồn dữ liệu:", DATA_URL)
print("Kích thước file:", os.path.getsize(SOURCE_FILE), "bytes")
print("Kích thước dataset:", df.shape)

display(df.head())


Nguồn dữ liệu: https://raw.githubusercontent.com/huyenpiano1711/BTQT2_Telco_dbt_BigQuery/main/telco_customer_churn.csv
Kích thước file: 977501 bytes
Kích thước dataset: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


Cell 4 — Nạp dữ liệu vào BigQuery Bronze

In [8]:

dataset_id = f"{PROJECT_ID}.{BRONZE_DATASET}"

# Schema tường minh: 21 cột STRING + metadata
schema = [bigquery.SchemaField(c, "STRING") for c in df.columns] + [
    bigquery.SchemaField("_ingested_at", "TIMESTAMP"),
    bigquery.SchemaField("_source_file", "STRING"),
]

bronze_df = df.copy()
bronze_df["_ingested_at"] = pd.Timestamp.now(tz="UTC")
bronze_df["_source_file"] = SOURCE_FILE

table_id = f"{dataset_id}.{BRONZE_TABLE}"
job = bq.load_table_from_dataframe(
    bronze_df, table_id,
    job_config=bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_TRUNCATE"),  # chạy lại không bị nhân đôi
)
job.result()
print("Đã nạp", bq.get_table(table_id).num_rows, "dòng vào", table_id)


Đã nạp 7043 dòng vào btqt2-telco-dbt-bigquery.bronze_telco.customers_raw


In [ ]:
# Thống kê các cột số
num = df[["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]].apply(pd.to_numeric, errors="coerce")
num.describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00,7032.00
mean,0.16,32.37,64.76,2283.30
std,0.37,24.56,30.09,2266.77
min,0.00,0.00,18.25,18.80
25%,0.00,9.00,35.50,401.45
50%,0.00,29.00,70.35,1397.48
75%,0.00,55.00,89.85,3794.74
max,1.00,72.00,118.75,8684.80


Cell 5 — Đối chiếu dữ liệu CSV và Bronze

In [9]:

q = f'''
SELECT
  COUNT(*)                                        AS so_dong,
  COUNT(DISTINCT customerID)                      AS so_customer_id,
  COUNTIF(TRIM(TotalCharges) = '')                AS total_charges_trong,
  ROUND(SUM(CAST(MonthlyCharges AS NUMERIC)), 2)  AS tong_monthly_charges,
  COUNTIF(Churn = 'Yes')                          AS churn_yes,
  COUNTIF(Churn = 'No')                           AS churn_no
FROM `{table_id}`
'''
r = bq.query(q).to_dataframe().iloc[0]
bronze_tbl = bq.get_table(table_id)

csv_stats = {
    "so_dong":              len(df),
    "so_customer_id":       df["customerID"].nunique(),
    "total_charges_trong":  (df["TotalCharges"].str.strip() == "").sum(),
    "tong_monthly_charges": round(df["MonthlyCharges"].astype(float).sum(), 2),
    "churn_yes":            (df["Churn"] == "Yes").sum(),
    "churn_no":             (df["Churn"] == "No").sum(),
}
doi_chieu = pd.DataFrame({
    "Chỉ số": list(csv_stats) + ["so_cot_nghiep_vu"],
    "CSV":    [float(v) for v in csv_stats.values()] + [df.shape[1]],
    "Bronze": [float(r[k]) for k in csv_stats] + [len([f for f in bronze_tbl.schema if not f.name.startswith("_")])],
})
doi_chieu["Kết quả"] = (doi_chieu["CSV"].round(2) == doi_chieu["Bronze"].round(2)).map({True: "Khớp", False: "Lệch"})
doi_chieu


,Chỉ số,CSV,Bronze,Kết quả
0,so_dong,7043.0,7043.0,Khớp
1,so_customer_id,7043.0,7043.0,Khớp
2,total_charges_trong,11.0,11.0,Khớp
3,tong_monthly_charges,456116.6,456116.6,Khớp
4,churn_yes,1869.0,1869.0,Khớp
5,churn_no,5174.0,5174.0,Khớp
6,so_cot_nghiep_vu,21.0,21.0,Khớp


Cell 6 — Đối chiếu customerID và kiểm tra NULL

In [10]:

ids_bq  = set(bq.query(f"SELECT customerID FROM `{table_id}`").to_dataframe()["customerID"])
ids_csv = set(df["customerID"])
print("Có trong CSV nhưng thiếu ở Bronze :", len(ids_csv - ids_bq))
print("Có ở Bronze nhưng không có trong CSV:", len(ids_bq - ids_csv))

null_expr = " + ".join(f"COUNTIF({c} IS NULL)" for c in df.columns)
print("Số ô NULL trên 21 cột ở Bronze    :", list(bq.query(f"SELECT {null_expr} AS n FROM `{table_id}`").result())[0].n)


Có trong CSV nhưng thiếu ở Bronze : 0
Có ở Bronze nhưng không có trong CSV: 0
Số ô NULL trên 21 cột ở Bronze    : 0


Cell 7 — Hiển thị schema và dữ liệu mẫu

In [11]:

# Minh chứng: schema (21 cột STRING + 2 cột metadata) và 10 dòng mẫu của Bronze
display(pd.DataFrame([(f.name, f.field_type, f.mode) for f in bronze_tbl.schema], columns=["column", "type", "mode"]))
bq.query(f"SELECT * FROM `{table_id}` LIMIT 10").to_dataframe()


,column,type,mode
0,customerID,STRING,NULLABLE
1,gender,STRING,NULLABLE
2,SeniorCitizen,STRING,NULLABLE
3,Partner,STRING,NULLABLE
4,Dependents,STRING,NULLABLE
5,tenure,STRING,NULLABLE
6,PhoneService,STRING,NULLABLE
7,MultipleLines,STRING,NULLABLE
8,InternetService,STRING,NULLABLE
9,OnlineSecurity,STRING,NULLABLE


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,_ingested_at,_source_file
0,3473-XIIIT,Female,0,Yes,No,16,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100,1534.75,Yes,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
1,8709-KRDVL,Female,0,No,No,33,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100,3320.6,No,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
2,8397-MVTAZ,Male,0,Yes,No,34,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3480,Yes,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
3,1666-JXLKU,Female,0,No,No,37,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3810.55,No,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
4,2669-OIDSD,Female,0,Yes,No,48,Yes,No,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.05,4834,No,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
5,2632-UCGVD,Male,1,Yes,No,66,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Credit card (automatic),100.05,6871.9,Yes,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
6,9647-ERGBE,Female,0,Yes,Yes,30,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,3046.15,Yes,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
7,8634-MPHTR,Male,1,Yes,No,47,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.05,4871.05,Yes,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
8,5515-AKOAJ,Female,0,No,No,54,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.1,5440.9,Yes,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
9,9094-AZPHK,Female,0,No,No,15,Yes,Yes,Fiber optic,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,100.15,1415,No,2026-10-13 06:50:49.047121+00:00,telco_customer_churn.csv
